In [7]:
from mGST.utility_functions_comparisons import get_full_mgst_parameters_from_configuration, factorize_psd_truncated
from iqm.benchmarks.compressive_gst.compressive_gst import GSTConfiguration

from mGST.low_level_jit import cost_function_numba, cost_function_jax_mps, gradient_k_mps_jit, gradient_k_numba, gradient_k_mps

import jax.numpy as jnp
import jax

import numpy as np

import matplotlib.pyplot as plt

backend = "iqmfakeapollo"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Check new functions using factorizations for $\rho$ and $E$ (POVM)

In [2]:
# let's start with a 2 qubit one to check with numba.
Q2_GST = GSTConfiguration(
    qubit_layouts=[[0, 1]],
    gate_set="2QXYCZ",
    num_circuits=800,
    shots=1000,
    rank=4,
)

K, X, E, rho, y, J, l, d, pdim, r, n_povm, bsize, meas_samples, n, nt, rK = get_full_mgst_parameters_from_configuration(
    Q2_GST, backend
)

2025-02-14 18:51:08,024 - iqm.benchmarks.logging_config - INFO - Now generating 800 random GST circuits...
2025-02-14 18:51:08,669 - iqm.benchmarks.logging_config - INFO - Will transpile all 800 circuits according to fixed physical layout
2025-02-14 18:51:08,669 - iqm.benchmarks.logging_config - INFO - Transpiling for backend IQMFakeApolloBackend with optimization level 0, sabre routing method all circuits
2025-02-14 18:51:10,390 - iqm.benchmarks.logging_config - INFO - Submitting batch with 800 circuits corresponding to qubits [0, 1]
2025-02-14 18:51:10,400 - iqm.benchmarks.logging_config - INFO - Now executing the corresponding circuit batch
2025-02-14 18:51:10,461 - iqm.benchmarks.logging_config - INFO - Retrieving all counts
INFO:2025-02-14 18:51:16,526:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
2025-02-14 18:51:16,526 - jax._src.xla_bridge - INFO - Unable to initialize backend 'rocm': module 'ja

In [4]:
K_np = np.array(K)
E_np = np.array(E)
rho_np = np.array(rho)
y_np = np.array(y)

In [6]:
cost_value_numba_2q = cost_function_numba(K_np, E_np, rho_np, J, y_np)
cost_value_numba_2q

0.0016613482049566656

In [129]:
# creating the parameters needed for the jax version
num_povm = E.shape[0]
dim = int(jnp.sqrt(E.shape[1]))

povm_tensor = jnp.reshape(E, shape=(num_povm, dim, dim))
state = jnp.reshape(rho, shape=(dim, dim))

indices_list = [indices[indices != -1] for indices in J]

kraus_tensor = K
prob_matrix = y

state_psd = factorize_psd_truncated(psd=state, max_rank=1)
jnp.allclose(state_psd @ state_psd.T, state)

Array(True, dtype=bool)

In [19]:
cost_value_jax_2q = cost_function_jax_mps(kraus_tensor, povm_tensor, state_psd, indices_list, prob_matrix)
jnp.allclose(cost_value_jax_2q, cost_value_numba_2q)

Array(True, dtype=bool)

In [ ]:
state_cholesky = jnp.linalg.cholesky(state)
jnp.allclose(state_cholesky @ state_cholesky.T.conj(), state)

Array(False, dtype=bool)

In [125]:
povm_psd  = factorize_psd_truncated(psd=povm_tensor, max_rank=1)
povm_psd.shape

(4, 4, 1)

In [126]:
jnp.allclose(jnp.einsum("ijk, ilk -> ijl", povm_psd, povm_psd.conj()), povm_tensor)

Array(True, dtype=bool)

In [131]:
povm_psd.shape

(4, 4, 1)

In [136]:
jnp.allclose(povm_psd @ povm_psd.conj().transpose(0, 2, 1), povm_tensor)

Array(True, dtype=bool)

In [147]:
cost_value_jax_2q_factorized_full = cost_function_jax_mps(kraus_tensor, povm_tensor, state_psd, indices_list, prob_matrix)
jnp.allclose(cost_value_jax_2q_factorized_full, cost_value_numba_2q)

Array(True, dtype=bool)

In [145]:
%time cost_function_jax_mps(kraus_tensor, povm_tensor, state_psd, indices_list, prob_matrix)

CPU times: user 858 ms, sys: 34.1 ms, total: 892 ms
Wall time: 765 ms


Array(0.00166135, dtype=float64)

In [146]:
%timeit cost_function_jax_mps(kraus_tensor, povm_tensor, state_psd, indices_list, prob_matrix)
# using the hadamard product:
# 726 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

726 ms ± 12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [139]:
jnp.einsum("ijk, jl, ilk -> i", povm_psd, state, povm_psd.conj())

Array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j], dtype=complex128)

In [140]:
jnp.einsum_path("ijk, jl, ilk -> i", povm_psd, state, povm_psd.conj())

([(0, 1), (0, 1)],
   Complete contraction:  ijk,jl,ilk->i
          Naive scaling:  4
      Optimized scaling:  4
       Naive FLOP count:  1.920e+2
   Optimized FLOP count:  1.600e+2
    Theoretical speedup:  1.200e+0
   Largest intermediate:  1.600e+1 elements
 --------------------------------------------------------------------------------
 scaling        BLAS                current                             remaining
 --------------------------------------------------------------------------------
    4           TDOT            jl,ijk->lik                            ilk,lik->i
    3              0             lik,ilk->i                                  i->i)